# Policy Regression Tests
Full end-to-end routing checks for all seven complaint policies.
Each cell runs a scripted answer sequence and prints:
- The question intent sequence (routing check)
- Final HPI fields
- Final policy_answers
- Final question_status
- Missing clarifications

No LLM is exercised — `submit_answer` uses rule-based parsing only for deterministic targets.

In [1]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow


def run_policy_test(session_name, primary_complaint, answers, label=None):
    """
    Drive the engine through a scripted answer sequence.
    Returns the final app state for inspection.
    Prints a compact summary of routing and final state.
    """
    label = label or session_name
    print("=" * 70)
    print(f"TEST: {label}")
    print("=" * 70)

    app = IntakeAppFlow(conn=None)
    app.new_session(
        session_name=session_name,
        primary_complaint=primary_complaint,
        auto_save=False,
    )

    first = app.start_intake(auto_save=False)
    intents = [first["intent"]]

    for answer in answers:
        result = app.submit_answer(answer, auto_save=False)
        intents.append(result["next_intent"])

    state = app.get_state()

    print("\nINTENT SEQUENCE:")
    for i, intent in enumerate(intents):
        print(f"  {i:>2}. {intent}")

    print("\nFINAL HPI:")
    pprint(state["hpi"])

    print("\nFINAL POLICY ANSWERS (non-None only):")
    pprint({k: v for k, v in state["policy_answers"].items() if v is not None})

    print("\nQUESTION STATUS:")
    pprint(state["conversation_meta"]["question_status"])

    print("\nMISSING CLARIFICATIONS:")
    pprint(state["missing_clarifications"])

    print("\nFLAGS:", state["flags"])
    print("INTAKE COMPLETE:", state["conversation_meta"]["intake_complete"])
    print()

    return app


print("Helper loaded.")

Helper loaded.


## 1. Headache — benign migraine-like presentation

In [2]:
run_policy_test(
    session_name="headache_regression",
    primary_complaint="headache",
    label="HEADACHE — benign migraine-like",
    answers=[
        # chief complaint
        "I have a headache.",
        # critical followups
        "No.",                                                      # sudden_severe_onset
        "No weakness, numbness, trouble speaking, or similar.",     # neurologic_symptoms
        "No vision changes.",                                       # visual_changes
        "No confusion.",                                            # confusion_or_ams
        "No fever or neck stiffness.",                              # fever_or_neck_stiffness
        "No recent head trauma.",                                   # head_trauma
        "No, I am not pregnant and not postpartum.",                # pregnancy_or_postpartum_context
        # must_characterize
        "This morning.",                                            # onset
        "About four hours.",                                        # duration
        "7 out of 10.",                                             # severity
        "It comes and goes.",                                       # timing
        "It has stayed about the same.",                            # course
        "In the front of my head.",                                 # location
        "Throbbing.",                                               # character
        "Bright light makes it worse.",                             # aggravating_factors
        "Rest helps a little.",                                     # relieving_factors
        "Nausea and sensitivity to light.",                         # associated_symptoms
        # high_priority_followups
        "Ibuprofen 400mg as needed.",                               # medications
        "No known allergies.",                                      # allergies
        "Yes, light sensitivity.",                                  # photophobia
        "No.",                                                      # phonophobia
        "Yes, nausea.",                                             # nausea_or_vomiting
        "No aura before this headache.",                            # aura_features
        "No, no exertion triggered it.",                            # exertional_trigger
        "No, position does not change it.",                         # positional_component
        "No, this is my usual headache pattern.",                   # new_or_progressive_pattern
        "No, I have not been overusing pain medicine.",             # medication_overuse_context
        # wrap-up
        "Nothing else.",
    ],
)

TEST: HEADACHE — benign migraine-like

INTENT SEQUENCE:
   0. ask_main_reason_for_visit
   1. ask_sudden_severe_onset
   2. ask_neurologic_symptoms
   3. ask_visual_changes
   4. ask_confusion_or_ams
   5. ask_fever_or_neck_stiffness
   6. ask_head_trauma
   7. ask_pregnancy_or_postpartum_context
   8. ask_onset
   9. ask_duration
  10. ask_severity
  11. ask_timing
  12. ask_course
  13. ask_location
  14. ask_character
  15. ask_aggravating_factors
  16. ask_relieving_factors
  17. ask_associated_symptoms
  18. ask_associated_symptoms
  19. ask_associated_symptoms
  20. ask_associated_symptoms
  21. ask_associated_symptoms
  22. ask_associated_symptoms
  23. ask_associated_symptoms
  24. ask_associated_symptoms
  25. ask_associated_symptoms
  26. ask_associated_symptoms
  27. ask_associated_symptoms
  28. ask_associated_symptoms
  29. ask_associated_symptoms

FINAL HPI:
{'aggravating_factors': ['bright light'],
 'associated_symptoms': [],
 'character': 'Throbbing.',
 'context': None,

## 2. Chest Pain — benign musculoskeletal presentation

In [3]:
run_policy_test(
    session_name="chest_pain_regression",
    primary_complaint="chest pain",
    label="CHEST PAIN — benign musculoskeletal",
    answers=[
        # chief complaint
        "I have chest pain.",
        # critical followups
        "No shortness of breath.",                                  # shortness_of_breath
        "No, I have not fainted or felt faint.",                    # syncope_or_presyncope
        "No, it is not getting worse quickly.",                     # rapid_worsening
        # must_characterize
        "This morning.",                                            # onset
        "Left side of my chest.",                                   # location
        "About two hours.",                                         # duration
        "5 out of 10.",                                             # severity
        "Sharp.",                                                   # character
        "Deep breathing makes it worse.",                           # aggravating_factors
        "Rest helps.",                                              # relieving_factors
        "No other symptoms.",                                       # associated_symptoms
        # high_priority_followups
        "Aspirin 81mg daily.",                                      # medications
        "Penicillin.",                                              # allergies
        "No, it does not move anywhere else.",                      # radiation
        "No nausea.",                                               # nausea
        "No sweating.",                                             # diaphoresis
        "No, it does not come on with activity.",                   # exertional_component
        # wrap-up
        "Nothing else.",
    ],
)

TEST: CHEST PAIN — benign musculoskeletal
EXTRACTOR FALLBACK: ValueError LLM did not return valid JSON: ```json
{
  "set_fields": {},
  "append_fields": [
    "hpi.relieving_factors": [
      "No other symptoms"
    ]
  ],
  "flags_to_add": [],
  "missing_clarifications_to_add": []
}
```

INTENT SEQUENCE:
   0. ask_main_reason_for_visit
   1. ask_shortness_of_breath
   2. ask_syncope_or_presyncope
   3. ask_rapid_worsening
   4. ask_onset
   5. ask_location
   6. ask_duration
   7. ask_severity
   8. ask_character
   9. ask_aggravating_factors
  10. ask_relieving_factors
  11. ask_relieving_factors
  12. ask_associated_symptoms
  13. ask_associated_symptoms
  14. ask_associated_symptoms
  15. ask_associated_symptoms
  16. ask_associated_symptoms
  17. ask_associated_symptoms
  18. ask_associated_symptoms
  19. ask_associated_symptoms

FINAL HPI:
{'aggravating_factors': ['deep breathing makes it worse.'],
 'associated_symptoms': [],
 'character': 'Sharp.',
 'context': None,
 'course': N

## 3. Abdominal Pain — benign gastroenteritis-like presentation

In [4]:
run_policy_test(
    session_name="abdominal_pain_regression",
    primary_complaint="abdominal pain",
    label="ABDOMINAL PAIN — benign gastroenteritis-like",
    answers=[
        # chief complaint
        "I have stomach pain.",
        # critical followups
        "No vomiting.",                                             # vomiting
        "No fever.",                                                # fever
        "No blood in my stool.",                                    # bloody_stool_or_melena
        "No, I am not pregnant.",                                   # pregnancy_or_postpartum_context
        "No, I have not fainted.",                                  # syncope_or_presyncope
        # must_characterize
        "Last night.",                                              # onset
        "About twelve hours.",                                      # duration
        "6 out of 10.",                                             # severity
        "It comes and goes.",                                       # timing
        "It has been about the same.",                              # course
        "Around my belly button.",                                  # location
        "Crampy.",                                                  # character
        "Eating makes it worse.",                                   # aggravating_factors
        "Nothing really helps.",                                    # relieving_factors
        "Nausea.",                                                  # associated_symptoms
        # high_priority_followups
        "No medications.",                                          # medications
        "No allergies.",                                            # allergies
        "Yes, some nausea.",                                        # nausea
        "Yes, loose stools.",                                       # diarrhea
        "No constipation.",                                         # constipation
        "No urinary symptoms.",                                     # urinary_symptoms
        "About six hours ago.",                                     # last_oral_intake
        # wrap-up
        "Nothing else.",
    ],
)

TEST: ABDOMINAL PAIN — benign gastroenteritis-like
EXTRACTOR FALLBACK: ValueError LLM did not return valid JSON: ```json
{
  "set_fields": {},
  "append_fields": [
    "hpi.relieving_factors\": [
      "Nothing really helps."
    ]
  ],
  "flags_to_add": [],
  "missing_clarifications_to_add": []
}
```

INTENT SEQUENCE:
   0. ask_main_reason_for_visit
   1. ask_vomiting
   2. ask_fever
   3. ask_bloody_stool_or_melena
   4. ask_pregnancy_or_postpartum_context
   5. ask_syncope_or_presyncope
   6. ask_onset
   7. ask_duration
   8. ask_severity
   9. ask_timing
  10. ask_course
  11. ask_location
  12. ask_character
  13. ask_aggravating_factors
  14. ask_relieving_factors
  15. ask_associated_symptoms
  16. ask_associated_symptoms
  17. ask_associated_symptoms
  18. ask_associated_symptoms
  19. ask_associated_symptoms
  20. ask_associated_symptoms
  21. ask_associated_symptoms
  22. ask_associated_symptoms
  23. ask_associated_symptoms
  24. ask_associated_symptoms

FINAL HPI:
{'aggrav

## 4. Shortness of Breath — benign exertional presentation

In [5]:
run_policy_test(
    session_name="sob_regression",
    primary_complaint="shortness of breath",
    label="SHORTNESS OF BREATH — benign exertional",
    answers=[
        # chief complaint
        "I have shortness of breath.",
        # critical followups
        "No chest pain.",                                           # chest_pain
        "No, it is not getting worse quickly.",                     # rapid_worsening
        "No, I have not fainted.",                                  # syncope_or_presyncope
        "No fever.",                                                # fever
        "No wheezing.",                                             # wheezing
        # must_characterize
        "This morning.",                                            # onset
        "About three hours.",                                       # duration
        "6 out of 10.",                                             # severity
        "It comes and goes.",                                       # timing
        "It has stayed about the same.",                            # course
        "Walking makes it worse.",                                  # aggravating_factors
        "Rest helps.",                                              # relieving_factors
        "Cough.",                                                   # associated_symptoms
        # high_priority_followups
        "Albuterol as needed.",                                     # medications
        "No allergies.",                                            # allergies
        "Yes, a cough.",                                            # cough
        "No, I can breathe fine lying flat.",                       # orthopnea
        "No leg swelling.",                                         # leg_swelling
        "Yes, activity makes it worse.",                            # exertional_component
        # wrap-up
        "Nothing else.",
    ],
)

TEST: SHORTNESS OF BREATH — benign exertional

INTENT SEQUENCE:
   0. ask_main_reason_for_visit
   1. ask_chest_pain
   2. ask_rapid_worsening
   3. ask_syncope_or_presyncope
   4. ask_fever
   5. ask_wheezing
   6. ask_onset
   7. ask_duration
   8. ask_severity
   9. ask_timing
  10. ask_course
  11. ask_aggravating_factors
  12. ask_relieving_factors
  13. ask_associated_symptoms
  14. ask_associated_symptoms
  15. ask_associated_symptoms
  16. ask_associated_symptoms
  17. ask_associated_symptoms
  18. ask_associated_symptoms
  19. ask_associated_symptoms
  20. ask_associated_symptoms
  21. ask_associated_symptoms

FINAL HPI:
{'aggravating_factors': ['walking makes it worse.'],
 'associated_symptoms': [],
 'character': None,
 'context': None,
 'course': 'It has stayed about the same.',
 'duration': 'about three hours.',
 'location': None,
 'onset': 'This morning.',
 'relieving_factors': ['rest helps.'],
 'severity': '6/10',
 'summary': None,
 'timing': 'It comes and goes.'}

FINAL 

## 5. Dizziness — benign positional presentation

In [6]:
run_policy_test(
    session_name="dizziness_regression",
    primary_complaint="dizziness",
    label="DIZZINESS — benign positional",
    answers=[
        # chief complaint
        "I have dizziness.",
        # critical followups
        "No, I have not fainted or felt like I might faint.",       # syncope_or_presyncope
        "No weakness, numbness, or trouble speaking.",              # neurologic_symptoms
        "No chest pain.",                                           # chest_pain
        "No shortness of breath.",                                  # shortness_of_breath
        "No recent head trauma.",                                   # head_trauma
        # must_characterize
        "This morning.",                                            # onset
        "About three hours.",                                       # duration
        "5 out of 10.",                                             # severity
        "It comes and goes.",                                       # timing
        "It has stayed about the same.",                            # course
        "Standing up makes it worse.",                              # aggravating_factors
        "Sitting down helps.",                                      # relieving_factors
        "Nausea.",                                                  # associated_symptoms
        # high_priority_followups
        "No medications.",                                          # medications
        "No allergies.",                                            # allergies
        "No vision changes.",                                       # visual_changes
        "No heart racing.",                                         # palpitations
        "Yes, it gets worse when I stand.",                         # positional_component
        "Yes, nausea.",                                             # nausea
        # wrap-up
        "Nothing else.",
    ],
)

TEST: DIZZINESS — benign positional
EXTRACTOR FALLBACK: ValueError LLM did not return valid JSON: ```json
{
  "set_fields": {},
  "append_fields": [
    "hpi.relieving_factors\": [
      \"Sitting down helps.\""
    ]
  ],
  "flags_to_add": [],
  "missing_clarifications_to_add": []
}
```

INTENT SEQUENCE:
   0. ask_main_reason_for_visit
   1. ask_syncope_or_presyncope
   2. ask_neurologic_symptoms
   3. ask_chest_pain
   4. ask_shortness_of_breath
   5. ask_head_trauma
   6. ask_onset
   7. ask_duration
   8. ask_severity
   9. ask_timing
  10. ask_course
  11. ask_aggravating_factors
  12. ask_relieving_factors
  13. ask_associated_symptoms
  14. ask_associated_symptoms
  15. ask_associated_symptoms
  16. ask_associated_symptoms
  17. ask_associated_symptoms
  18. ask_associated_symptoms
  19. ask_associated_symptoms
  20. ask_associated_symptoms
  21. ask_associated_symptoms

FINAL HPI:
{'aggravating_factors': ['standing up makes it worse.'],
 'associated_symptoms': [],
 'character':

## 6. Sore Throat — benign viral presentation

In [7]:
run_policy_test(
    session_name="sore_throat_regression",
    primary_complaint="sore throat",
    label="SORE THROAT — benign viral",
    answers=[
        # chief complaint
        "I have a sore throat.",
        # critical followups
        "No shortness of breath.",                                  # shortness_of_breath
        "No trouble swallowing.",                                   # trouble_swallowing
        "No drooling.",                                             # drooling
        "No fever.",                                                # fever
        # must_characterize
        "Yesterday evening.",                                       # onset
        "About sixteen hours.",                                     # duration
        "4 out of 10.",                                             # severity
        "It is constant.",                                          # timing
        "It has been getting slightly worse.",                      # course
        "Scratchy and raw.",                                        # character
        "Swallowing makes it worse.",                               # aggravating_factors
        "Cold drinks help a little.",                               # relieving_factors
        "Runny nose and mild cough.",                               # associated_symptoms
        # high_priority_followups
        "No medications.",                                          # medications
        "No allergies.",                                            # allergies
        "Yes, a mild cough.",                                       # cough
        "No voice change.",                                         # voice_change
        "Yes, my partner has a cold.",                              # sick_contacts
        # wrap-up
        "Nothing else.",
    ],
)

TEST: SORE THROAT — benign viral
EXTRACTOR FALLBACK: ValueError LLM did not return valid JSON: ```json
{
  "set_fields": {},
  "append_fields": [
    "hpi.relieving_factors\": [
      \"Cold drinks help a little.\"
    ]"
  ],
  "flags_to_add": [],
  "missing_clarifications_to_add": []
}
```

INTENT SEQUENCE:
   0. ask_main_reason_for_visit
   1. ask_shortness_of_breath
   2. ask_trouble_swallowing
   3. ask_drooling
   4. ask_fever
   5. ask_onset
   6. ask_duration
   7. ask_severity
   8. ask_timing
   9. ask_course
  10. ask_character
  11. ask_aggravating_factors
  12. ask_relieving_factors
  13. ask_associated_symptoms
  14. ask_associated_symptoms
  15. ask_associated_symptoms
  16. ask_associated_symptoms
  17. ask_associated_symptoms
  18. ask_associated_symptoms
  19. ask_associated_symptoms
  20. ask_associated_symptoms

FINAL HPI:
{'aggravating_factors': ['swallowing makes it worse.'],
 'associated_symptoms': [],
 'character': 'Scratchy and raw.',
 'context': None,
 'course

## 7. Back Pain — benign musculoskeletal presentation

In [8]:
run_policy_test(
    session_name="back_pain_regression",
    primary_complaint="back pain",
    label="BACK PAIN — benign musculoskeletal",
    answers=[
        # chief complaint
        "I have low back pain.",
        # critical followups
        "No weakness or numbness in my legs.",                      # neurologic_symptoms
        "No bowel or bladder changes.",                             # bowel_or_bladder_changes
        "No fever.",                                                # fever
        "No recent head trauma.",                                   # head_trauma
        "No, it is not getting worse quickly.",                     # rapid_worsening
        # must_characterize
        "This morning.",                                            # onset
        "About six hours.",                                         # duration
        "6 out of 10.",                                             # severity
        "It is constant.",                                          # timing
        "It has been about the same.",                              # course
        "Lower back, right side.",                                  # location
        "Dull and achy.",                                           # character
        "Bending makes it worse.",                                  # aggravating_factors
        "Lying down helps a little.",                               # relieving_factors
        "No other symptoms.",                                       # associated_symptoms
        # high_priority_followups
        "Ibuprofen as needed.",                                     # medications
        "No allergies.",                                            # allergies
        "No, it does not go down my leg.",                          # radiation
        "Yes, I lifted something heavy yesterday.",                 # recent_heavy_lifting
        "No numbness or tingling.",                                 # numbness_or_tingling
        # wrap-up
        "Nothing else.",
    ],
)

TEST: BACK PAIN — benign musculoskeletal

INTENT SEQUENCE:
   0. ask_main_reason_for_visit
   1. ask_neurologic_symptoms
   2. ask_bowel_or_bladder_changes
   3. ask_fever
   4. ask_head_trauma
   5. ask_rapid_worsening
   6. ask_onset
   7. ask_duration
   8. ask_severity
   9. ask_timing
  10. ask_course
  11. ask_location
  12. ask_character
  13. ask_aggravating_factors
  14. end_intake_early
  15. end_intake_early
  16. end_intake_early
  17. end_intake_early
  18. end_intake_early
  19. end_intake_early
  20. end_intake_early
  21. end_intake_early
  22. end_intake_early

FINAL HPI:
{'aggravating_factors': [],
 'associated_symptoms': [],
 'character': 'Dull and achy.',
 'context': None,
 'course': 'It has been about the same.',
 'duration': 'about six hours.',
 'location': 'Lower back, right side.',
 'onset': 'This morning.',
 'relieving_factors': [],
 'severity': '6/10',
 'summary': None,
 'timing': 'It is constant.'}

FINAL POLICY ANSWERS (non-None only):
{'aura_features': [],


## 8. Escalation — Chest Pain with Shortness of Breath (red flag trigger)

In [9]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

print("=" * 70)
print("TEST: ESCALATION — chest pain + shortness of breath flag")
print("=" * 70)

app = IntakeAppFlow(conn=None)
app.new_session(
    session_name="escalation_chest_sob",
    primary_complaint="chest pain",
    auto_save=False,
)

app.start_intake(auto_save=False)

# Patient reports chest pain as chief complaint
r1 = app.submit_answer("I have chest pain.", auto_save=False)
print("After chief complaint — next intent:", r1["next_intent"])

# Patient reports shortness of breath → should trigger flag
r2 = app.submit_answer("Yes, I have shortness of breath.", auto_save=False)
print("After SOB answer — next intent:", r2["next_intent"])

state = app.get_state()
print("\nFLAGS:", state["flags"])
print("ACUITY SIGNALS:", state["conversation_meta"]["acuity_signal"])
print("Expected flag: chest_pain_with_shortness_of_breath")
print("Expected next intent: escalate_high_risk_chest_pain or recommend_immediate_attention")

TEST: ESCALATION — chest pain + shortness of breath flag
After chief complaint — next intent: ask_shortness_of_breath
After SOB answer — next intent: ask_syncope_or_presyncope

FLAGS: []
ACUITY SIGNALS: []
Expected flag: chest_pain_with_shortness_of_breath
Expected next intent: escalate_high_risk_chest_pain or recommend_immediate_attention


## 9. Stop / Decline / Unknown — edge case handling

In [10]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow

print("=" * 70)
print("TEST: EDGE CASES — stop, skip, unknown")
print("=" * 70)

app = IntakeAppFlow(conn=None)
app.new_session(
    session_name="edge_case_test",
    primary_complaint="headache",
    auto_save=False,
)
app.start_intake(auto_save=False)

# Get to aggravating_factors
for answer in [
    "I have a headache.",
    "No.",
    "No weakness, numbness, trouble speaking, or similar.",
    "No vision changes.",
    "No confusion.",
    "No fever or neck stiffness.",
    "No recent head trauma.",
    "No, I am not pregnant.",
    "This morning.",
    "About four hours.",
    "7 out of 10.",
    "It comes and goes.",
    "It has stayed about the same.",
    "In the front of my head.",
    "Throbbing.",
]:
    app.submit_answer(answer, auto_save=False)

# Test: Skip (decline)
r_skip = app.submit_answer("Skip", auto_save=False)
print("SKIP result:")
print("  current_intent:", r_skip["current_intent"])
print("  next_intent:", r_skip["next_intent"])
print("  question_status aggravating_factors:",
      app.get_state()["conversation_meta"]["question_status"].get("aggravating_factors"))
print("  Expected: asked_declined")
print()

# Test: Unknown
r_unknown = app.submit_answer("I don't know", auto_save=False)
print("UNKNOWN result (relieving_factors):")
print("  current_intent:", r_unknown["current_intent"])
print("  next_intent:", r_unknown["next_intent"])
print("  question_status relieving_factors:",
      app.get_state()["conversation_meta"]["question_status"].get("relieving_factors"))
print("  Expected: asked_unknown")
print()

# Test: Stop
r_stop = app.submit_answer("I want to stop", auto_save=False)
print("STOP result:")
print("  next_intent:", r_stop["next_intent"])
print("  user_requested_stop:",
      app.get_state()["conversation_meta"]["user_requested_stop"])
print("  intake_complete:",
      app.get_state()["conversation_meta"]["intake_complete"])
print("  Expected next intent: end_intake_early")

TEST: EDGE CASES — stop, skip, unknown
SKIP result:
  current_intent: ask_aggravating_factors
  next_intent: ask_relieving_factors
  question_status aggravating_factors: asked_declined
  Expected: asked_declined

UNKNOWN result (relieving_factors):
  current_intent: ask_relieving_factors
  next_intent: ask_associated_symptoms
  question_status relieving_factors: asked_unknown
  Expected: asked_unknown

STOP result:
  next_intent: end_intake_early
  user_requested_stop: True
  intake_complete: True
  Expected next intent: end_intake_early
